In [2]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Document classification started")

Document classification started


In [3]:
train_df = pd.read_csv("../processed/train.csv")
val_df = pd.read_csv("../processed/validation.csv")
test_df = pd.read_csv("../processed/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (4646, 4)
Validation: (996, 4)
Test: (996, 4)


In [4]:
X_train = train_df["model_text"]
y_train = train_df["label"]

X_val = val_df["model_text"]
y_val = val_df["label"]

X_test = test_df["model_text"]
y_test = test_df["label"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

Training samples: 4646
Validation samples: 996
Test samples: 996


In [5]:
vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (4646, 30000)
Validation TF-IDF shape: (996, 30000)
Test TF-IDF shape: (996, 30000)


In [6]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_tfidf, y_train)

print("Model training completed")

Model training completed


In [7]:
y_val_pred = model.predict(X_val_tfidf)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

Validation Accuracy: 0.9979919678714859

Classification Report:
                precision    recall  f1-score   support

      Contract       0.99      1.00      0.99        76
         Email       1.00      1.00      1.00       300
       Invoice       1.00      1.00      1.00       300
Purchase Order       1.00      0.95      0.97        20
        Report       1.00      1.00      1.00       300

      accuracy                           1.00       996
     macro avg       1.00      0.99      0.99       996
  weighted avg       1.00      1.00      1.00       996



In [8]:
y_test_pred = model.predict(X_test_tfidf)

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.9969879518072289

Classification Report:
                precision    recall  f1-score   support

      Contract       1.00      0.99      0.99        77
         Email       1.00      1.00      1.00       300
       Invoice       1.00      1.00      1.00       300
Purchase Order       1.00      0.95      0.97        19
        Report       1.00      1.00      1.00       300

      accuracy                           1.00       996
     macro avg       1.00      0.99      0.99       996
  weighted avg       1.00      1.00      1.00       996



In [9]:
cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=model.classes_
)

cm_df = pd.DataFrame(
    cm,
    index=model.classes_,
    columns=model.classes_
)

print(cm_df)

                Contract  Email  Invoice  Purchase Order  Report
Contract              76      1        0               0       0
Email                  0    299        0               0       1
Invoice                0      0      300               0       0
Purchase Order         0      0        1              18       0
Report                 0      0        0               0     300


In [10]:
errors = test_df[y_test != y_test_pred].copy()

errors["predicted_label"] = y_test_pred[y_test != y_test_pred]

print("Number of errors:", len(errors))

for i in range(len(errors)):
    print("\n" + "=" * 80)
    print("Document ID:", errors.iloc[i]["document_id"])
    print("Actual:", errors.iloc[i]["label"])
    print("Predicted:", errors.iloc[i]["predicted_label"])
    print("Text preview:")
    print(errors.iloc[i]["model_text"][:1000])

Number of errors: 3

Document ID: contract_0485
Actual: Contract
Predicted: Email
Text preview:
Exhibit 99.1 JOINT FILING AGREEMENT Additional Reporting Person (a): Merck Serono SA Address: Zone Industrielle 1267 Coinsins, Switzerland Additional Reporting Person (b): Merck KGaA Address: Frankfurter Str. 250 64293 Darmstadt, Germany Designated Filer: Ares Trading SA Issuer and CUSIP: Intrexon Corporation (46122T102) Dated: January 7, 2019 ARES TRADING SA ARES TRADING SA By: /s/ Cédric Hyde By: /s/ Luigia Bocola Name: Cédric Hyde Name: Luigia Bocola Title: CFO Title: Finance Manager MERCK SERONO SA, COINSINS, SWITZERLAND, AN AFFILIATE OF MERCK KGAA, DARMSTADT, GERMANY MERCK SERONO SA, COINSINS, SWITZERLAND, AN AFFILIATE OF MERCK KGAA, DARMSTADT, GERMANY By: /s/ Cédric Hyde By: /s/ Tearaboth Te Name: Cédric Hyde Name: Tearaboth Te Title: CFO Title: Treasury Director MERCK KGAA, DARMSTADT, GERMANY MERCK KGAA, DARMSTADT, GERMANY By: /s/ Rando Bruns By: /s/ Tim Nielsen Name: Rando Bruns Name

In [11]:
error_indices = test_df.index[y_test != y_test_pred]

error_probs = model.predict_proba(
    vectorizer.transform(test_df.loc[error_indices, "model_text"])
)

for i in range(len(error_indices)):
    idx = error_indices[i]

    probabilities = pd.Series(
        error_probs[i],
        index=model.classes_
    ).sort_values(ascending=False)

    print("\n" + "=" * 70)
    print("Document:", test_df.loc[idx, "document_id"])
    print("Actual:", test_df.loc[idx, "label"])
    print("Predicted:", y_test_pred[list(test_df.index).index(idx)])
    print(probabilities)


Document: contract_0485
Actual: Contract
Predicted: Email
Email             0.338674
Invoice           0.312232
Contract          0.164793
Report            0.097407
Purchase Order    0.086894
dtype: float64

Document: po_0105
Actual: Purchase Order
Predicted: Invoice
Invoice           0.435710
Purchase Order    0.325224
Contract          0.121087
Email             0.078935
Report            0.039043
dtype: float64

Document: email_0848
Actual: Email
Predicted: Report
Report            0.708511
Email             0.137069
Invoice           0.074992
Contract          0.059492
Purchase Order    0.019936
dtype: float64


In [12]:
feature_names = vectorizer.get_feature_names_out()

for class_index, class_name in enumerate(model.classes_):
    coefficients = model.coef_[class_index]

    top_indices = coefficients.argsort()[-15:][::-1]

    print("\n" + "=" * 60)
    print("CLASS:", class_name)

    for i in top_indices:
        print(feature_names[i], round(coefficients[i], 3))


CLASS: Contract
agreement 2.629
shall 2.335
this agreement 2.122
parties 1.516
shall be 1.475
party 1.331
the parties 1.329
exhibit 1.25
any 0.985
such 0.965
agree 0.958
of the 0.942
effective 0.935
agreement this 0.929
between 0.883

CLASS: Email
phillip 2.958
you 2.43
enron 2.08
the 1.785
allen 1.629
ect 1.452
phillip allen 1.43
com 1.386
your 1.288
hou 1.234
hou ect 1.215
2001 1.199
enron com 1.147
if you 1.116
here 1.105

CLASS: Invoice
invoice 3.822
00 3.25
total 1.66
amount 1.526
due 1.45
box 1.272
date 1.207
tobacco 1.085
99 1.042
invoice date 1.024
1999 1.014
fax 0.977
payment 0.945
ny 0.936
invoice no 0.89

CLASS: Purchase Order
purchase 1.997
order 1.959
purchase order 1.752
vendor 1.629
buyer 1.459
requisitioner 1.432
6a 1.398
ship to 1.306
requisition 1.277
ship 1.121
60 radio 1.11
phone 1.108
freight 1.107
radio 1.094
this order 1.066

CLASS: Report
our 1.716
td 1.644
td td 1.43
tr 1.28
growth 1.131
of our 1.128
year 1.124
we 1.101
tr td 1.088
td tr 1.087
tr tr 1.051
annu

In [13]:
import re

def clean_model_text(text):
    text = str(text)

    # Remove HTML tags such as <td>, <tr>, etc.
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove Markdown image references
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)

    # Remove page split markers
    text = text.replace("<--- Page Split --->", " ")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


train_df["model_text_clean"] = train_df["model_text"].apply(clean_model_text)
val_df["model_text_clean"] = val_df["model_text"].apply(clean_model_text)
test_df["model_text_clean"] = test_df["model_text"].apply(clean_model_text)

print(train_df["model_text_clean"].head())

0    # UNITED STATES SECURITIES AND EXCHANGE COMMIS...
1    Jacques, Would it be ok if I signed new consul...
2    Invoice / Affidavit OHANA MEDIA GROUP, LLC. 83...
3    FW: Enron' s August Baseload Physical Fixed Pr...
4    Please Remit To: KNZZ News Radio 1100 1360 E. ...
Name: model_text_clean, dtype: str


In [14]:
X_train_clean = train_df["model_text_clean"]
y_train = train_df["label"]

X_val_clean = val_df["model_text_clean"]
y_val = val_df["label"]

X_test_clean = test_df["model_text_clean"]
y_test = test_df["label"]

In [15]:
vectorizer_clean = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf_clean = vectorizer_clean.fit_transform(X_train_clean)
X_val_tfidf_clean = vectorizer_clean.transform(X_val_clean)
X_test_tfidf_clean = vectorizer_clean.transform(X_test_clean)

print("Train:", X_train_tfidf_clean.shape)
print("Validation:", X_val_tfidf_clean.shape)
print("Test:", X_test_tfidf_clean.shape)

Train: (4646, 30000)
Validation: (996, 30000)
Test: (996, 30000)


In [16]:
model_clean = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model_clean.fit(X_train_tfidf_clean, y_train)

y_val_pred_clean = model_clean.predict(X_val_tfidf_clean)
y_test_pred_clean = model_clean.predict(X_test_tfidf_clean)

print("Validation Accuracy:",
      accuracy_score(y_val, y_val_pred_clean))

print("Test Accuracy:",
      accuracy_score(y_test, y_test_pred_clean))

Validation Accuracy: 0.9979919678714859
Test Accuracy: 0.9959839357429718


In [17]:
errors_clean = test_df[
    y_test != y_test_pred_clean
].copy()

errors_clean["predicted_label"] = y_test_pred_clean[
    y_test != y_test_pred_clean
]

print("Number of errors:", len(errors_clean))

for i in range(len(errors_clean)):
    print("\n" + "=" * 80)
    print("Document:", errors_clean.iloc[i]["document_id"])
    print("Actual:", errors_clean.iloc[i]["label"])
    print("Predicted:", errors_clean.iloc[i]["predicted_label"])
    print(errors_clean.iloc[i]["model_text_clean"][:1000])

Number of errors: 4

Document: contract_0485
Actual: Contract
Predicted: Email
Exhibit 99.1 JOINT FILING AGREEMENT Additional Reporting Person (a): Merck Serono SA Address: Zone Industrielle 1267 Coinsins, Switzerland Additional Reporting Person (b): Merck KGaA Address: Frankfurter Str. 250 64293 Darmstadt, Germany Designated Filer: Ares Trading SA Issuer and CUSIP: Intrexon Corporation (46122T102) Dated: January 7, 2019 ARES TRADING SA ARES TRADING SA By: /s/ Cédric Hyde By: /s/ Luigia Bocola Name: Cédric Hyde Name: Luigia Bocola Title: CFO Title: Finance Manager MERCK SERONO SA, COINSINS, SWITZERLAND, AN AFFILIATE OF MERCK KGAA, DARMSTADT, GERMANY MERCK SERONO SA, COINSINS, SWITZERLAND, AN AFFILIATE OF MERCK KGAA, DARMSTADT, GERMANY By: /s/ Cédric Hyde By: /s/ Tearaboth Te Name: Cédric Hyde Name: Tearaboth Te Title: CFO Title: Treasury Director MERCK KGAA, DARMSTADT, GERMANY MERCK KGAA, DARMSTADT, GERMANY By: /s/ Rando Bruns By: /s/ Tim Nielsen Name: Rando Bruns Name: Tim Nielsen Tit

In [20]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training will use CPU")

PyTorch version: 2.14.0+cpu
CUDA available: False
Training will use CPU


In [21]:
!nvidia-smi

Fri Sep 18 19:17:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.62                 KMD Version: 610.62        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   78C    P8              4W /   45W |     175MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
import subprocess

result = subprocess.run(
    ["powershell", "-Command", "Get-CimInstance Win32_VideoController | Select-Object Name"],
    capture_output=True,
    text=True
)

print(result.stdout)


Name                                  
----                                  
Intel(R) UHD Graphics                 
NVIDIA GeForce RTX 3050 6GB Laptop GPU





In [23]:
%pip uninstall -y torch

Found existing installation: torch 2.14.0
Uninstalling torch-2.14.0:
  Successfully uninstalled torch-2.14.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [24]:
%pip install torch==2.14.0 --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
   ---------------------------------------- 0.0/2.0 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 GB 3.7 MB/s eta 0:08:54
   ---------------------------------------- 0.0/2.0 GB 5.2 MB/s eta 0:06:20
   ---------------------------------------- 0.0/2.0 GB 5.3 MB/s eta 0:06:12
   ---------------------------------------- 0.0/2.0 GB 5.9 MB/s eta 0:05:37
   ---------------------------------------- 0.0/2.0 GB 6.6 MB/s eta 0:05:02
   ---------------------------------------- 0.0/2.0 GB 7.7 MB/s eta 0:04:17
   ---------------------------------------- 0.0/2.0 GB 8.1 MB/s eta 0:04:06
   ---------------------------------------- 0.0/2.0 GB 9.0 MB/s eta 0:03:41
   ---------------------------------------- 0.0/2.0 GB 9.5 MB/s eta 0:03:28
   ---------------------------------------- 0.0/2.0 GB 9.4 MB/s eta 0:03:31
   ----------------------------------------

In [25]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cpu
CUDA available: False


In [26]:
import sys
import subprocess

print("Python executable:")
print(sys.executable)

print("\nTorch installation:")
subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "show",
    "torch"
])

Python executable:
c:\Projects\DocuMind\ml\.venv\Scripts\python.exe

Torch installation:


CompletedProcess(args=['c:\\Projects\\DocuMind\\ml\\.venv\\Scripts\\python.exe', '-m', 'pip', 'show', 'torch'], returncode=0)

In [27]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "torch"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Name: torch
Version: 2.14.0+cu130
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-Clause AND BSD-3-Clause AND BSL-1.0 AND MIT
Location: c:\Projects\DocuMind\ml\.venv\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: 




In [28]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

PyTorch: 2.14.0+cpu
CUDA available: False


In [29]:
import sys
import torch
import subprocess

print("Notebook Python:")
print(sys.executable)

print("\nTorch imported by notebook:")
print(torch.__file__)
print("Version:", torch.__version__)

print("\nTorch from a fresh Python process:")
result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch; print(torch.__file__); print(torch.__version__); print(torch.cuda.is_available())"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Notebook Python:
c:\Projects\DocuMind\ml\.venv\Scripts\python.exe

Torch imported by notebook:
c:\Projects\DocuMind\ml\.venv\Lib\site-packages\torch\__init__.py
Version: 2.14.0+cpu

Torch from a fresh Python process:
c:\Projects\DocuMind\ml\.venv\Lib\site-packages\torch\__init__.py
2.14.0+cu130
True




In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import torch

gpu = torch.cuda.get_device_properties(0)

print("GPU:", gpu.name)
print("VRAM:", round(gpu.total_memory / (1024**3), 2), "GB")

GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.0 GB


In [3]:
%pip install transformers accelerate

  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl (2.9 MB)
Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)
Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl (355 kB)
Using cached typer-0.27.2-py3-none-any.whl (123 kB)
Using cached ann

In [4]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Projects\DocuMind\ml\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sudee\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded
Vocabulary size: 30522


In [5]:
from transformers import AutoModelForSequenceClassification

labels = sorted(train_df["label"].unique())

label2id = {
    label: i
    for i, label in enumerate(labels)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Labels:", labels)
print("Label mapping:", label2id)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded")
print("Device:", device)

NameError: name 'train_df' is not defined

In [6]:
import pandas as pd

train_df = pd.read_csv("../processed/train.csv")
val_df = pd.read_csv("../processed/validation.csv")
test_df = pd.read_csv("../processed/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (4646, 4)
Validation: (996, 4)
Test: (996, 4)


In [7]:
from transformers import AutoModelForSequenceClassification

labels = sorted(train_df["label"].unique())

label2id = {
    label: i
    for i, label in enumerate(labels)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Labels:", labels)
print("Label mapping:", label2id)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded")
print("Device:", device)

Labels: ['Contract', 'Email', 'Invoice', 'Purchase Order', 'Report']
Label mapping: {'Contract': 0, 'Email': 1, 'Invoice': 2, 'Purchase Order': 3, 'Report': 4}


c:\Projects\DocuMind\ml\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sudee\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded
Device: cuda


In [8]:
from datasets import Dataset

train_hf = Dataset.from_pandas(
    train_df[["model_text_clean", "label"]],
    preserve_index=False
)

val_hf = Dataset.from_pandas(
    val_df[["model_text_clean", "label"]],
    preserve_index=False
)

test_hf = Dataset.from_pandas(
    test_df[["model_text_clean", "label"]],
    preserve_index=False
)

print(train_hf)
print(val_hf)
print(test_hf)

KeyError: "['model_text_clean'] not in index"

In [9]:
import re

def clean_model_text(text):
    text = str(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove Markdown images
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)

    # Remove page split markers
    text = text.replace("<--- Page Split --->", " ")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


train_df["model_text_clean"] = train_df["model_text"].apply(clean_model_text)
val_df["model_text_clean"] = val_df["model_text"].apply(clean_model_text)
test_df["model_text_clean"] = test_df["model_text"].apply(clean_model_text)

print(train_df[["model_text", "model_text_clean"]].head())

                                          model_text  \
0  # UNITED STATES SECURITIES AND EXCHANGE COMMIS...   
1  Jacques, Would it be ok if I signed new consul...   
2  Invoice / Affidavit OHANA MEDIA GROUP, LLC. 83...   
3  FW: Enron' s August Baseload Physical Fixed Pr...   
4  Please Remit To: KNZZ News Radio 1100 1360 E. ...   

                                    model_text_clean  
0  # UNITED STATES SECURITIES AND EXCHANGE COMMIS...  
1  Jacques, Would it be ok if I signed new consul...  
2  Invoice / Affidavit OHANA MEDIA GROUP, LLC. 83...  
3  FW: Enron' s August Baseload Physical Fixed Pr...  
4  Please Remit To: KNZZ News Radio 1100 1360 E. ...  


In [10]:
from datasets import Dataset

train_hf = Dataset.from_pandas(
    train_df[["model_text_clean", "label"]],
    preserve_index=False
)

val_hf = Dataset.from_pandas(
    val_df[["model_text_clean", "label"]],
    preserve_index=False
)

test_hf = Dataset.from_pandas(
    test_df[["model_text_clean", "label"]],
    preserve_index=False
)

print(train_hf)
print(val_hf)
print(test_hf)

Dataset({
    features: ['model_text_clean', 'label'],
    num_rows: 4646
})
Dataset({
    features: ['model_text_clean', 'label'],
    num_rows: 996
})
Dataset({
    features: ['model_text_clean', 'label'],
    num_rows: 996
})


In [11]:
train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

print(train_df[["label", "label_id"]].drop_duplicates().sort_values("label_id"))

             label  label_id
12        Contract         0
1            Email         1
2          Invoice         2
16  Purchase Order         3
0           Report         4


In [12]:
train_hf = Dataset.from_pandas(
    train_df[["model_text_clean", "label_id"]],
    preserve_index=False
)

val_hf = Dataset.from_pandas(
    val_df[["model_text_clean", "label_id"]],
    preserve_index=False
)

test_hf = Dataset.from_pandas(
    test_df[["model_text_clean", "label_id"]],
    preserve_index=False
)

print(train_hf)
print(val_hf)
print(test_hf)

Dataset({
    features: ['model_text_clean', 'label_id'],
    num_rows: 4646
})
Dataset({
    features: ['model_text_clean', 'label_id'],
    num_rows: 996
})
Dataset({
    features: ['model_text_clean', 'label_id'],
    num_rows: 996
})


In [13]:
def tokenize_function(batch):
    return tokenizer(
        batch["model_text_clean"],
        truncation=True,
        max_length=512
    )


train_tokenized = train_hf.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_hf.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_hf.map(
    tokenize_function,
    batched=True
)

print(train_tokenized)
print(val_tokenized)
print(test_tokenized)

Map:   0%|          | 0/4646 [00:00<?, ? examples/s]

Map:   0%|          | 0/996 [00:00<?, ? examples/s]

Map:   0%|          | 0/996 [00:00<?, ? examples/s]

Dataset({
    features: ['model_text_clean', 'label_id', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4646
})
Dataset({
    features: ['model_text_clean', 'label_id', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 996
})
Dataset({
    features: ['model_text_clean', 'label_id', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 996
})


In [14]:
token_lengths = train_tokenized["input_ids"]

lengths = pd.Series([len(x) for x in token_lengths])

print(lengths.describe())
print("\nExactly 512 tokens:", (lengths == 512).sum())
print("Less than 512 tokens:", (lengths < 512).sum())

count    4646.000000
mean      388.815110
std       156.661107
min        40.000000
25%       247.000000
50%       512.000000
75%       512.000000
max       512.000000
dtype: float64

Exactly 512 tokens: 2496
Less than 512 tokens: 2150


In [15]:
train_lengths = pd.Series(
    [len(x) for x in train_tokenized["input_ids"]]
)

train_check = pd.DataFrame({
    "label": train_df["label"].values,
    "tokens": train_lengths.values
})

class_token_stats = train_check.groupby("label")["tokens"].agg(
    documents="count",
    avg_tokens="mean",
    truncated=lambda x: (x == 512).sum(),
    truncated_percent=lambda x: (x == 512).mean() * 100
)

print(class_token_stats)

                documents  avg_tokens  truncated  truncated_percent
label                                                              
Contract              357  507.890756        347          97.198880
Email                1400  264.847857        267          19.071429
Invoice              1400  361.109286        459          32.785714
Purchase Order         89  369.235955         25          28.089888
Report               1400  511.368571       1398          99.857143


In [16]:
def create_chunks(df, tokenizer, max_length=512, stride=128):
    rows = []

    for i in range(len(df)):
        text = df.iloc[i]["model_text_clean"]

        encoded = tokenizer(
            text,
            max_length=max_length,
            truncation=True,
            stride=stride,
            return_overflowing_tokens=True
        )

        for chunk_id in range(len(encoded["input_ids"])):
            rows.append({
                "document_id": df.iloc[i]["document_id"],
                "chunk_id": chunk_id,
                "label": df.iloc[i]["label"],
                "label_id": df.iloc[i]["label_id"],
                "input_ids": encoded["input_ids"][chunk_id],
                "attention_mask": encoded["attention_mask"][chunk_id]
            })

    return pd.DataFrame(rows)


train_chunks = create_chunks(train_df, tokenizer)
val_chunks = create_chunks(val_df, tokenizer)
test_chunks = create_chunks(test_df, tokenizer)

print("Train chunks:", train_chunks.shape)
print("Validation chunks:", val_chunks.shape)
print("Test chunks:", test_chunks.shape)

Train chunks: (7142, 6)
Validation chunks: (1547, 6)
Test chunks: (1537, 6)


In [17]:
print("Train chunks by class:")
print(train_chunks["label"].value_counts())

print("\nValidation chunks by class:")
print(val_chunks["label"].value_counts())

print("\nTest chunks by class:")
print(test_chunks["label"].value_counts())

Train chunks by class:
label
Report            2798
Invoice           1859
Email             1667
Contract           704
Purchase Order     114
Name: count, dtype: int64

Validation chunks by class:
label
Report            600
Invoice           404
Email             367
Contract          148
Purchase Order     28
Name: count, dtype: int64

Test chunks by class:
label
Report            600
Invoice           402
Email             361
Contract          151
Purchase Order     23
Name: count, dtype: int64


In [18]:
chunk_stats = train_chunks.groupby("label").agg(
    documents=("document_id", "nunique"),
    chunks=("chunk_id", "count")
)

chunk_stats["chunks_per_document"] = (
    chunk_stats["chunks"] / chunk_stats["documents"]
)

print(chunk_stats)

                documents  chunks  chunks_per_document
label                                                 
Contract              357     704             1.971989
Email                1400    1667             1.190714
Invoice              1400    1859             1.327857
Purchase Order         89     114             1.280899
Report               1400    2798             1.998571


In [19]:
train_chunk_hf = Dataset.from_pandas(
    train_chunks[
        ["input_ids", "attention_mask", "label_id"]
    ],
    preserve_index=False
)

val_chunk_hf = Dataset.from_pandas(
    val_chunks[
        ["input_ids", "attention_mask", "label_id"]
    ],
    preserve_index=False
)

test_chunk_hf = Dataset.from_pandas(
    test_chunks[
        ["input_ids", "attention_mask", "label_id"]
    ],
    preserve_index=False
)

print(train_chunk_hf)
print(val_chunk_hf)
print(test_chunk_hf)

Dataset({
    features: ['input_ids', 'attention_mask', 'label_id'],
    num_rows: 7142
})
Dataset({
    features: ['input_ids', 'attention_mask', 'label_id'],
    num_rows: 1547
})
Dataset({
    features: ['input_ids', 'attention_mask', 'label_id'],
    num_rows: 1537
})


In [20]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Dynamic padding enabled")

Dynamic padding enabled


In [21]:
train_chunk_hf = train_chunk_hf.rename_column("label_id", "labels")
val_chunk_hf = val_chunk_hf.rename_column("label_id", "labels")
test_chunk_hf = test_chunk_hf.rename_column("label_id", "labels")

print(train_chunk_hf)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7142
})


In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/distilbert_doc_classifier",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    fp16=True,

    logging_steps=50,

    report_to="none"
)

print("Training configuration ready")

Training configuration ready


In [23]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = predictions.argmax(axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

print("Metrics ready")

Metrics ready


In [24]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_chunk_hf,
    eval_dataset=val_chunk_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready")

Trainer ready


In [25]:
import torch

print("GPU:", torch.cuda.get_device_name(0))

free_memory, total_memory = torch.cuda.mem_get_info()

print("Total VRAM:", round(total_memory / (1024**3), 2), "GB")
print("Free VRAM:", round(free_memory / (1024**3), 2), "GB")

GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Total VRAM: 6.0 GB
Free VRAM: 4.75 GB


In [26]:
print("Starting DistilBERT training...")

trainer.train()

print("Training completed.")

Starting DistilBERT training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.160366,0.071461,0.980608,0.947333
2,0.031962,0.052980,0.987072,0.968351


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed.


In [27]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

test_predictions = trainer.predict(test_chunk_hf)

logits = test_predictions.predictions

prediction_df = pd.DataFrame(
    logits,
    columns=[f"logit_{i}" for i in range(len(labels))]
)

prediction_df["document_id"] = test_chunks["document_id"].values
prediction_df["true_label"] = test_chunks["label"].values

document_logits = prediction_df.groupby("document_id")[
    [f"logit_{i}" for i in range(len(labels))]
].mean()

document_true = prediction_df.groupby("document_id")["true_label"].first()

document_pred_ids = document_logits.values.argmax(axis=1)

document_predictions = pd.Series(
    [id2label[i] for i in document_pred_ids],
    index=document_logits.index
)

print("Number of test documents:", len(document_predictions))

print("\nDocument-level Accuracy:",
      accuracy_score(document_true, document_predictions))

print("\nDocument-level Classification Report:")
print(
    classification_report(
        document_true,
        document_predictions
    )
)

Number of test documents: 996

Document-level Accuracy: 0.9979919678714859

Document-level Classification Report:
                precision    recall  f1-score   support

      Contract       1.00      1.00      1.00        77
         Email       1.00      1.00      1.00       300
       Invoice       1.00      1.00      1.00       300
Purchase Order       1.00      0.95      0.97        19
        Report       1.00      1.00      1.00       300

      accuracy                           1.00       996
     macro avg       1.00      0.99      0.99       996
  weighted avg       1.00      1.00      1.00       996



In [28]:
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="../models/distilbert_doc_classifier",

    num_train_epochs=4,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=True,

    logging_steps=50,
    report_to="none"
)

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_chunk_hf,
    eval_dataset=val_chunk_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("Trainer configured with early stopping")

Trainer configured with early stopping


In [30]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)

model.to(device)

print("Fresh model loaded")
print("Device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh model loaded
Device: cuda:0


In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_chunk_hf,
    eval_dataset=val_chunk_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("Trainer ready for controlled training")

Trainer ready for controlled training


In [32]:
trainer.train()


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_chunk_hf,
    eval_dataset=val_chunk_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("Trainer ready")

Trainer ready


In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.104781,0.049163,0.985779,0.967649
2,0.027681,0.057585,0.989011,0.978502
3,0.001208,0.048589,0.990950,0.980656
4,0.001040,0.044750,0.990950,0.981145


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1788, training_loss=0.0690950140652874, metrics={'train_runtime': 1068.4417, 'train_samples_per_second': 26.738, 'train_steps_per_second': 1.673, 'total_flos': 3751754023535280.0, 'train_loss': 0.0690950140652874, 'epoch': 4.0})

In [35]:
history = pd.DataFrame(trainer.state.log_history)

print(
    history[
        [
            "epoch",
            "loss",
            "eval_loss",
            "eval_accuracy",
            "eval_macro_f1"
        ]
    ].dropna(subset=["epoch"])
)

       epoch      loss  eval_loss  eval_accuracy  eval_macro_f1
0   0.111982  0.329242        NaN            NaN            NaN
1   0.223964  0.287676        NaN            NaN            NaN
2   0.335946  0.188241        NaN            NaN            NaN
3   0.447928  0.261850        NaN            NaN            NaN
4   0.559910  0.164293        NaN            NaN            NaN
5   0.671892  0.111256        NaN            NaN            NaN
6   0.783875  0.223458        NaN            NaN            NaN
7   0.895857  0.104781        NaN            NaN            NaN
8   1.000000       NaN   0.049163       0.985779       0.967649
9   1.006719  0.130602        NaN            NaN            NaN
10  1.118701  0.085394        NaN            NaN            NaN
11  1.230683  0.049374        NaN            NaN            NaN
12  1.342665  0.038307        NaN            NaN            NaN
13  1.454647  0.042138        NaN            NaN            NaN
14  1.566629  0.009185        NaN       

In [36]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation Macro F1:", trainer.state.best_metric)

test_results = trainer.evaluate(
    test_chunk_hf
)

print("\nTest Results:")
print(test_results)

Best checkpoint: ../models/distilbert_doc_classifier\checkpoint-1788
Best validation Macro F1: 0.9811446163184561


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.001040,0.018634,4,0.995446,0.996419



Test Results:
{'eval_loss': 0.01863424852490425, 'eval_accuracy': 0.9954456733897202, 'eval_macro_f1': 0.9964188313939353}


In [37]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Predict all test chunks using the best trained model
test_output = trainer.predict(test_chunk_hf)

logits = test_output.predictions

# Put chunk predictions into a DataFrame
chunk_predictions = pd.DataFrame(
    logits,
    columns=[f"logit_{i}" for i in range(len(labels))]
)

chunk_predictions["document_id"] = test_chunks["document_id"].values
chunk_predictions["true_label"] = test_chunks["label"].values

# Average chunk logits to get one prediction per document
document_logits = chunk_predictions.groupby("document_id")[
    [f"logit_{i}" for i in range(len(labels))]
].mean()

document_true = (
    chunk_predictions
    .groupby("document_id")["true_label"]
    .first()
)

# Convert logits to final document labels
document_pred_ids = document_logits.values.argmax(axis=1)

document_pred = pd.Series(
    [id2label[i] for i in document_pred_ids],
    index=document_logits.index
)

# Evaluate
document_accuracy = accuracy_score(
    document_true,
    document_pred
)

document_macro_f1 = f1_score(
    document_true,
    document_pred,
    average="macro"
)

print("Document-level Accuracy:", document_accuracy)
print("Document-level Macro F1:", document_macro_f1)

print("\nClassification Report:")
print(
    classification_report(
        document_true,
        document_pred
    )
)

Document-level Accuracy: 0.998995983935743
Document-level Macro F1: 0.9993333314814763

Classification Report:
                precision    recall  f1-score   support

      Contract       1.00      1.00      1.00        77
         Email       1.00      1.00      1.00       300
       Invoice       1.00      1.00      1.00       300
Purchase Order       1.00      1.00      1.00        19
        Report       1.00      1.00      1.00       300

      accuracy                           1.00       996
     macro avg       1.00      1.00      1.00       996
  weighted avg       1.00      1.00      1.00       996



In [38]:
train_results = trainer.evaluate(train_chunk_hf)

print("Training Accuracy:", train_results["eval_accuracy"])
print("Training Macro F1:", train_results["eval_macro_f1"])
print("Training Loss:", train_results["eval_loss"])

print("\nValidation Accuracy:",
      trainer.state.log_history[-2]["eval_accuracy"]
      if "eval_accuracy" in trainer.state.log_history[-2]
      else "See epoch logs")

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.001040,0.004610,4,0.999160,0.995124


Training Accuracy: 0.9991598991879026
Training Macro F1: 0.9951238735277768
Training Loss: 0.004610096570104361

Validation Accuracy: 0.9954456733897202


In [39]:
val_results = trainer.evaluate(val_chunk_hf)

print("Validation Loss:", val_results["eval_loss"])
print("Validation Accuracy:", val_results["eval_accuracy"])
print("Validation Macro F1:", val_results["eval_macro_f1"])

print("\nTraining vs Validation")
print("--------------------------------")
print("Training Accuracy :", train_results["eval_accuracy"])
print("Validation Accuracy:", val_results["eval_accuracy"])
print()
print("Training Macro F1  :", train_results["eval_macro_f1"])
print("Validation Macro F1:", val_results["eval_macro_f1"])

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.001040,0.044750,4,0.990950,0.981145


Validation Loss: 0.04475010186433792
Validation Accuracy: 0.9909502262443439
Validation Macro F1: 0.9811446163184561

Training vs Validation
--------------------------------
Training Accuracy : 0.9991598991879026
Validation Accuracy: 0.9909502262443439

Training Macro F1  : 0.9951238735277768
Validation Macro F1: 0.9811446163184561


In [40]:
val_output = trainer.predict(val_chunk_hf)

val_logits = val_output.predictions

val_prediction_df = pd.DataFrame(
    val_logits,
    columns=[f"logit_{i}" for i in range(len(labels))]
)

val_prediction_df["document_id"] = val_chunks["document_id"].values
val_prediction_df["true_label"] = val_chunks["label"].values

val_document_logits = val_prediction_df.groupby("document_id")[
    [f"logit_{i}" for i in range(len(labels))]
].mean()

val_document_true = (
    val_prediction_df
    .groupby("document_id")["true_label"]
    .first()
)

val_document_pred_ids = val_document_logits.values.argmax(axis=1)

val_document_pred = pd.Series(
    [id2label[i] for i in val_document_pred_ids],
    index=val_document_logits.index
)

print(
    "Document-level Validation Accuracy:",
    accuracy_score(val_document_true, val_document_pred)
)

print(
    "Document-level Validation Macro F1:",
    f1_score(
        val_document_true,
        val_document_pred,
        average="macro"
    )
)

Document-level Validation Accuracy: 0.9969879518072289
Document-level Validation Macro F1: 0.9922312701441209


In [41]:
from pathlib import Path
import json

model_dir = Path("../models/distilbert_doc_classifier/final")
model_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

with open(model_dir / "label_mapping.json", "w") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": id2label
        },
        f,
        indent=4
    )

print("Model saved to:", model_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ..\models\distilbert_doc_classifier\final
